# igittigitt Quickstart

Install the package, then try the API and CLI examples below.

## 1) Install

In [ ]:
import sys
from pathlib import Path


def checkout_root() -> Path | None:
    """The checkout this notebook runs in, if any.

    Installing it rather than the pushed default branch is what makes CI and a
    local run test the code under test.
    """
    for folder in [Path.cwd(), *Path.cwd().parents]:
        pyproject = folder / "pyproject.toml"
        text = pyproject.read_text(encoding="utf-8") if pyproject.is_file() else ""
        if 'name = "igittigitt"' in text.splitlines():
            return folder
    return None


# Install uv into this kernel's environment first: a bare `!uv` needs uv on
# PATH, which a reader running this notebook from a plain Python will not have.
!{sys.executable} -m pip install --quiet --upgrade uv
assert _exit_code == 0

# install igittigitt: this checkout when the notebook runs inside one (CI, a clone),
# else the default branch from GitHub
root = checkout_root()
source = str(root) if root is not None else "git+https://github.com/bitranox/igittigitt.git"
!{sys.executable} -m uv pip install --python {sys.executable} --upgrade {source}
assert _exit_code == 0

## 2) Library: ignore mode with a negation

In [ ]:
import igittigitt

parser = igittigitt.IgnoreParser()
parser.add_rule("*.py[cod]", base_path="/opt/project")
parser.add_rule("!keep.pyc", base_path="/opt/project")

print("main.pyc ignored:", parser.match("/opt/project/main.pyc"))
print("keep.pyc ignored:", parser.match("/opt/project/keep.pyc"))

## 3) Library: include / whitelist mode

In [ ]:
inc = igittigitt.IncludeParser()
inc.add_rule("*.py", base_path="/opt/project")
print("deep .py kept:", inc.match("/opt/project/pkg/deep.py"))

## 4) Library: spaces, backslashes and comments in rule lines

A rule line is read exactly as git reads it: only the final run of unescaped spaces is cut, a backslash escapes the next character, tabs are never cut, leading spaces belong to the pattern, and only a `#` in the first column starts a comment.

In [ ]:
base = "/opt/project"


def matches(rule: str, name: str) -> bool:
    """Whether this one rule, on its own, ignores the name."""
    parser = igittigitt.IgnoreParser()
    parser.add_rule(rule, base_path=base)
    return parser.match(f"{base}/{name}")


assert matches("*.log   ", "app.log")  # trailing spaces are cut
assert matches(r"name\ ", "name ")  # an escaped space stays
assert not matches(r"name\ ", "name")
assert matches(r"two \    ", "two  ")  # the plain space before the escape stays too
assert matches(r"pair \\\\ ", "pair \\\\")  # two escaped backslashes, then a cut space
assert matches("tab\t", "tab\t")  # a tab is never cut
assert not matches("tab\t", "tab")
assert matches("  *.log", "  x.log")  # leading spaces belong to the pattern
assert not matches("  *.log", "x.log")
assert matches("  # not a comment", "  # not a comment")  # only column-1 # is a comment
assert matches(r"\#hash", "#hash")  # escape a leading #
print("every rule-line example holds")

## 5) CLI: version, help and info

In [ ]:
!igittigitt --version
assert _exit_code == 0
!igittigitt --help
assert _exit_code == 0
!igittigitt info
assert _exit_code == 0